# Lab 11. Using Oracle 23ai to Store Embeddings and Similarity Search

## **Objective:** 

- Getting Started with Oracle 23ai

## Sequence 1. Getting Started with Oracle 23ai
### Import Required Packages
- You can see the LoadProperties.py, config.txt, faq.txt (inside txt-docs folder) and oci-ai-foundations.pdf (inside pdf-docs folder) with few other files in the File Browser section on the left-hand side.
- LoadProperties.py file defines a Python class named LoadProperties that reads configuration settings from a config.txt file and provides methods to access those settings
- LoadProperties() class reads the configuration values of model_name, endpoint, compartment_ocid, embedding_model_name from the config.txt file.

In [ ]:
from pypdf import PdfReader
import oracledb
import oci
from langchain.text_splitter import CharacterTextSplitter
from langchain_community.vectorstores.oraclevs import OracleVS
from langchain_community.embeddings import OCIGenAIEmbeddings
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_core.documents import BaseDocumentTransformer, Document
from LoadProperties import LoadProperties

# Setup basic variables
properties = LoadProperties()

print("Successfully imported libraries and modules")

In [ ]:
CONFIG_PROFILE = "DEFAULT"
config = oci.config.from_file('~/.oci/config', CONFIG_PROFILE)

# Service endpoint
endpoint = properties.getEndpoint()

In [ ]:
# Initialize the Generative AI Client

GenAI_client = oci.generative_ai_inference.GenerativeAiInferenceClient(config=config, service_endpoint=endpoint, retry_strategy=oci.retry.NoneRetryStrategy(), timeout=(10,240))

-	We set up a chat detail object and a chat request object for passing the user query and other model parameters.
-	`embed_text_detail` is an instance of instance of `EmbedTextDetails` which holds all the necessary details for the embedding request.
-	`embed_text_detail.inputs` assigns the text inputs for which embeddings need to be generated.
-	`embed_text_detail.truncate` specifies how text inputs should be truncated if they exceed the model’s maximum token length. "NONE" means that no truncation should occur, and the input is expected to fit within the model’s maximum token limit.

In [ ]:
# Set up Embedding Request

inputs = ["France"]
embed_text_detail = oci.generative_ai_inference.models.EmbedTextDetails()
embed_text_detail.serving_mode = oci.generative_ai_inference.models.OnDemandServingMode(
  model_id=properties.getEmbeddingModelName()
)
embed_text_detail.inputs = inputs
embed_text_detail.truncate = "NONE"
embed_text_detail.compartment_id = properties.getCompartment()

-	`embed_text()`method sends the `embed_text_detail` object to the OCI Generative AI service, which processes the input text and returns a vector representation (embedding).
-	`embed_text_response` stores the response from the AI model, which contains the generated embeddings.

**Note:**

-	You will see the list of vectors that represent the input text. Each number in the vector corresponds to a feature that the model has identified for "France."

In [ ]:
# Make the Embedding Request and Retrieve the Response
embed_text_response = GenAI_client.embed_text(embed_text_detail)

# Print first 10 Embeddings from the long list of 1024 Embeddings
print("******************Embed Texts Result******************")
print(embed_text_response.data.embeddings[0][:10])

## Sequence 2. RAG using Oracle Database 23ai - Vectorization


## Connect ot Oracle 23ai Container

- `oracledb.connect()`function establishes a connection to an Oracle database using the provided credentials and connection details. 
- The function takes username, password and dsn.
- The DSN (data source name) specifies the host, port and database service name to connect to. 
- You can refer the bash script init-genailabs.sh.

In [ ]:
! sudo podman stop 23ai
! sudo podman start 23ai

*Wait for 80 Seconds. Oracle 23ai Take some time to be available.*

In [ ]:
! sleep 80

In [ ]:
# Connect to the Oracle Database 23ai
un = "vector"
pw = "vector"
cs = "localhost/FREEPDB1"
# import oracledb
connection = oracledb.connect(user=un, password=pw, dsn=cs)
try:
    conn23c = oracledb.connect(user=un, password=pw, dsn=cs)
    print("Connection successful!")
except Exception as e:
    print("Connection failed!")

### Step 1 - Load the document and create pdf reader object

In [ ]:
pdf = PdfReader('./pdf-docs/oci-ai-foundations.pdf')

### Step 2 - Transform the document to text

In [ ]:
text=""
for page in pdf.pages:
    text += page.extract_text()
print("You have transformed the PDF document to text format")

### Step 3 - Chunk the text document into smaller chunks

In [ ]:
text_splitter = CharacterTextSplitter(separator=".",chunk_size=2000,chunk_overlap=100)
chunks = text_splitter.split_text(text)

### Function to format and add metadata to Oracle 23ai Vector Store

In [ ]:
def chunks_to_docs_wrapper(row: dict) -> Document:
    """
    Converts text into a Document object suitable for ingestion into Oracle Vector Store.
    - row (dict): A dictionary representing a row of data with keys for 'id', 'link', and 'text'.
    """
    metadata = {'id': row['id'], 'link': row['link']}
    return Document(page_content=row['text'], metadata=metadata)

### Step 4 - Create metadata wrapper to store additional information in the vector store

In [ ]:
"""
Converts a row from a DataFrame into a Document object suitable for ingestion into Oracle Vector Store.
- row (dict): A dictionary representing a row of data with keys for 'id', 'link', and 'text'.
"""
docs = [chunks_to_docs_wrapper({'id': str(page_num), 'link': f'Page {page_num}', 'text': text}) for page_num, text in enumerate(chunks)]

### Step 5 - Using an embedding model, embed the chunks as vectors into Oracle Database 23ai.

In [ ]:
embed_model = OCIGenAIEmbeddings(
    model_id=properties.getEmbeddingModelName(),
    service_endpoint=properties.getEndpoint(),
    compartment_id=properties.getCompartment(),
)

### Step 6 - Configure the vector store with the model, table name, and vectorize the chunks

In [ ]:
knowledge_base = OracleVS.from_documents(docs, embed_model, client=conn23c, table_name="DEMO_TABLE", distance_strategy=DistanceStrategy.DOT_PRODUCT)

print("Chunks are stored in the DEMO_TABLE")

# Sequence 3. Retrieve relevant documents and send these as a context in a query with ConversationalRetrievalChain chain.

In [ ]:
import oracledb
import oci
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_community.vectorstores.oraclevs import OracleVS
from langchain_community.chat_models.oci_generative_ai import ChatOCIGenAI
from langchain_community.embeddings import OCIGenAIEmbeddings
from LoadProperties import LoadProperties

print("Successfully imported libraries and modules")

-	`oracledb.connect()`function establishes a connection to an Oracle database using the provided credentials and connection details. 
-  The function takes username, password and dsn. The DSN (data source name) specifies the host, port and database service name to connect to.

In [ ]:
# Setup basic variables
properties = LoadProperties()

# Connect to the Oracle Database 23ai 
un = "vector"
pw = "vector"
cs = "localhost/FREEPDB1" 
import oracledb
connection = oracledb.connect(user=un, password=pw, dsn=cs)

# Connect to the database
try:
    conn23c = oracledb.connect(user=un, password=pw, dsn=cs)
    print("Connection successful!")
except Exception as e:
    print("Connection failed!")

### Step 1. Initialize embeddings and Chat models with OCI GenAI

In [ ]:
# Initialize embeddings model with OCI GenAI
embed_model = OCIGenAIEmbeddings(
    model_id=properties.getEmbeddingModelName(),
    service_endpoint=properties.getEndpoint(),
    compartment_id=properties.getCompartment(),
)

# Create vector store with Oracle database
vs = OracleVS(
    embedding_function=embed_model,
    client=conn23c,
    table_name="DEMO_TABLE",
    distance_strategy=DistanceStrategy.DOT_PRODUCT
)

# Set up retriever with similarity search
retv = vs.as_retriever(search_type="similarity", search_kwargs={'k': 3})

# Initialize the LLM with OCI GenAI
llm = ChatOCIGenAI(
    model_id=properties.getModelName(),
    service_endpoint=properties.getEndpoint(),
    compartment_id=properties.getCompartment(),
    model_kwargs={"max_tokens": 200}
)


### Step 2. Build the conversational retrieval chain

In [ ]:
# Create conversation memory
memory = ConversationBufferMemory(
    llm=llm,
    memory_key="chat_history",
    return_messages=True,
    output_key='answer'
)

# Build the conversational retrieval chain
qa = ConversationalRetrievalChain.from_llm(
    llm,
    retriever=retv,
    memory=memory,
    return_source_documents=True
)

# qa is now ready to use for conversational QA interactions

### Step 3. Setup a Query and Create the LLM Prompt 

In [ ]:
question="""Tell me something about OCI Foundations Course. Respond in English Language Only."""

In [ ]:
from IPython.display import display, Markdown
response = qa.invoke({"question": question})
ai_message = response["chat_history"][-1]
display(Markdown(ai_message.content))

In [ ]:
prompt = f"""  You are a helpful assistant. USE ONLY the sources below and ABSOLUTELY IGNORE any previous knowledge. Assume the customer is highly technical.
   Respond to PRECISELY to this question: "{question}.", USING ONLY the following information and IGNORING ANY PREVIOUS KNOWLEDGE.
   =====
   Answer (Three paragraphs, maximum 150 words each.):
   """

### Step 4. Call the OCI Generative AI Chat model and store the chat model’s response in `chat_response`.

In [ ]:
import oci
from LoadProperties import LoadProperties

# Setup basic variables 
properties = LoadProperties()

# Use Instance Principals for Authentication 
signer = oci.auth.signers.InstancePrincipalsSecurityTokenSigner()

generative_ai_inference_client = oci.generative_ai_inference.GenerativeAiInferenceClient(config=config, service_endpoint=properties.getEndpoint(), retry_strategy=oci.retry.NoneRetryStrategy(), timeout=(10,240))
chat_detail = oci.generative_ai_inference.models.ChatDetails()
chat_request = oci.generative_ai_inference.models.CohereChatRequest()
chat_request.message = prompt 
chat_request.max_tokens = 1000
chat_request.temperature = 0.0
chat_request.frequency_penalty = 0
chat_request.top_p = 0.75
chat_request.top_k = 0

chat_detail.serving_mode = oci.generative_ai_inference.models.OnDemandServingMode(model_id= properties.getModelName())
chat_detail.chat_request = chat_request 
chat_detail.compartment_id = properties.getCompartment() 
chat_response = generative_ai_inference_client.chat(chat_detail)


In [ ]:
response=chat_response.data.chat_response.chat_history[1].message

### Step 5. Print the Response

In [ ]:
display(Markdown(response))

## Congratulations! You have successfully completed this Lab.